In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "eclare").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from eclare.data.screen import fetch_human_mouse_tissue_metadata

c:\Users\home\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path
import duckdb

project_root = Path(r"C:\Projects\eclare")
SCREEN_DIR = project_root / "data" / "processed" / "screen_tissue"

HUMAN_DIR = SCREEN_DIR / "grch38-21564c80a8c9ebaf"
MOUSE_DIR = SCREEN_DIR / "mm10-647e6942a349e062"

PAIRS_PATH = SCREEN_DIR / "orthologous_els_baseline.parquet"
OUTPUT_PATH = SCREEN_DIR / "orthologous_els_tissue_baseline.parquet"

def parquet_glob(path):
    return (path / "*.parquet").as_posix()

con = duckdb.connect()

print("Human shards:", len(list(HUMAN_DIR.glob("*.parquet"))))
print("Mouse shards:", len(list(MOUSE_DIR.glob("*.parquet"))))
print("Файл пар найден:", PAIRS_PATH.exists())

Human shards: 101
Mouse shards: 106
Файл пар найден: True


In [4]:
def parquet_glob(path: Path) -> str:
    return (path / "*.parquet").as_posix()


con = duckdb.connect()

con.execute(f"""
COPY (
    WITH

    human AS (
        SELECT
            ccre_id AS human_id,
            tissue,
            assay,
            MAX(zscore) AS human_zscore,
            COUNT(*) AS human_n_measurements
        FROM read_parquet('{parquet_glob(HUMAN_DIR)}')
        WHERE tissue IS NOT NULL
        GROUP BY ccre_id, tissue, assay
    ),

    mouse AS (
        SELECT
            ccre_id AS mouse_id,
            tissue,
            assay,
            MAX(zscore) AS mouse_zscore,
            COUNT(*) AS mouse_n_measurements
        FROM read_parquet('{parquet_glob(MOUSE_DIR)}')
        WHERE tissue IS NOT NULL
        GROUP BY ccre_id, tissue, assay
    ),

    pairs AS (
        SELECT
            human_id,
            mouse_id
        FROM read_parquet('{PAIRS_PATH.as_posix()}')
    )

    SELECT
        p.human_id,
        p.mouse_id,
        h.tissue,
        h.assay,

        h.human_zscore,
        m.mouse_zscore,

        h.human_n_measurements,
        m.mouse_n_measurements

    FROM pairs p

    JOIN human h
        ON h.human_id = p.human_id

    JOIN mouse m
        ON m.mouse_id = p.mouse_id
       AND m.tissue = h.tissue
       AND m.assay = h.assay

    ORDER BY
        p.human_id,
        p.mouse_id,
        h.tissue,
        h.assay
)
TO '{OUTPUT_PATH.as_posix()}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

print("Saved to:", OUTPUT_PATH)

Saved to: C:\Projects\eclare\data\processed\screen_tissue\orthologous_els_tissue_baseline.parquet


In [5]:
import pandas as pd

In [7]:
df = pd.read_parquet(OUTPUT_PATH)
df.head()

,human_id,mouse_id,tissue,assay,human_zscore,mouse_zscore,human_n_measurements,mouse_n_measurements
0,EH38E0048077,EM10E1439220,adipose,DNase,1.008100,0.994911,3,2
1,EH38E0048077,EM10E1439220,adipose,H3K27ac,-0.018593,-0.221053,2,1
2,EH38E0048077,EM10E1439220,adipose,H3K4me3,0.474738,0.458568,5,1
3,EH38E0048077,EM10E1439220,adrenal gland,DNase,0.950425,1.379826,14,14
4,EH38E0048077,EM10E1439220,blood,ATAC,1.644916,0.412561,209,4


In [8]:
len(df)

1827360

In [9]:
keys = ["human_id", "mouse_id", "tissue"]

values = [
    "human_zscore",
    "mouse_zscore",
    "human_n_measurements",
    "mouse_n_measurements",
]

In [10]:
wide = df.pivot(
    index=keys,
    columns="assay",
    values=values,
)

wide.head()

human_zscore                      \
assay                                           ATAC      CTCF     DNase   
human_id     mouse_id     tissue                                           
EH38E0048077 EM10E1439220 adipose                NaN       NaN  1.008100   
                          adrenal gland          NaN       NaN  0.950425   
                          blood             1.644916  2.003889  1.353530   
                          bone marrow       0.725278  0.999385  0.848175   
                          brain             0.897288  1.258093  1.065049   

                                                            mouse_zscore  \
assay                                     H3K27ac   H3K4me3         ATAC   
human_id     mouse_id     tissue                                           
EH38E0048077 EM10E1439220 adipose       -0.018593  0.474738          NaN   
                          adrenal gland       NaN       NaN          NaN   
                          blood          1.340279  1.355181     0.412561   
                          bone marrow    1.302290  0.547427     0.534195   
                          brain          1.156469  0.994539     0.024305   

                                                                       \
assay                                        CTCF     DNase   H3K27ac   
human_id     mouse_id     tissue                                        
EH38E0048077 EM10E1439220 adipose             NaN  0.994911 -0.221053   
                          adrenal gland       NaN  1.379826       NaN   
                          blood          0.652838  0.906703  0.453939   
                          bone marrow    1.439177  0.591713  0.874432   
                          brain          0.561160  1.180892  0.917485   

                                                  human_n_measurements        \
assay                                     H3K4me3                 ATAC  CTCF   
human_id     mouse_id     tissue                                               
EH38E0048077 EM10E1439220 adipose        0.458568                  NaN   NaN   
                          adrenal gland       NaN                  NaN   NaN   
                          blood          0.460114                209.0  25.0   
                          bone marrow    0.548215                  1.0   4.0   
                          brain          0.923420                 10.0  69.0   

                                                                \
assay                                    DNase H3K27ac H3K4me3   
human_id     mouse_id     tissue                                 
EH38E0048077 EM10E1439220 adipose          3.0     2.0     5.0   
                          adrenal gland   14.0     NaN     NaN   
                          blood          401.0   152.0   163.0   
                          bone marrow     24.0    13.0    18.0   
                          brain          245.0    87.0   102.0   

                                        mouse_n_measurements             \
assay                                                   ATAC CTCF DNase   
human_id     mouse_id     tissue                                          
EH38E0048077 EM10E1439220 adipose                        NaN  NaN   2.0   
                          adrenal gland                  NaN  NaN  14.0   
                          blood                          4.0  1.0   8.0   
                          bone marrow                    7.0  2.0   3.0   
                          brain                         23.0  7.0  80.0   

                                                         
assay                                   H3K27ac H3K4me3  
human_id     mouse_id     tissue                         
EH38E0048077 EM10E1439220 adipose           1.0     1.0  
                          adrenal gland     NaN     NaN  
                          blood             1.0     1.0  
                          bone marrow       2.0     4.0  
                          brain            27.0    27.0

In [11]:
wide.columns = [
    f"{name}_{assay}"
    for name, assay in wide.columns
]

wide = wide.reset_index()
wide.head()

,human_id,mouse_id,tissue,human_zscore_ATAC,human_zscore_CTCF,human_zscore_DNase,human_zscore_H3K27ac,human_zscore_H3K4me3,mouse_zscore_ATAC,mouse_zscore_CTCF,...,human_n_measurements_ATAC,human_n_measurements_CTCF,human_n_measurements_DNase,human_n_measurements_H3K27ac,human_n_measurements_H3K4me3,mouse_n_measurements_ATAC,mouse_n_measurements_CTCF,mouse_n_measurements_DNase,mouse_n_measurements_H3K27ac,mouse_n_measurements_H3K4me3
0,EH38E0048077,EM10E1439220,adipose,NaN,NaN,1.008100,-0.018593,0.474738,NaN,NaN,...,NaN,NaN,3.0,2.0,5.0,NaN,NaN,2.0,1.0,1.0
1,EH38E0048077,EM10E1439220,adrenal gland,NaN,NaN,0.950425,NaN,NaN,NaN,NaN,...,NaN,NaN,14.0,NaN,NaN,NaN,NaN,14.0,NaN,NaN
2,EH38E0048077,EM10E1439220,blood,1.644916,2.003889,1.353530,1.340279,1.355181,0.412561,0.652838,...,209.0,25.0,401.0,152.0,163.0,4.0,1.0,8.0,1.0,1.0
3,EH38E0048077,EM10E1439220,bone marrow,0.725278,0.999385,0.848175,1.302290,0.547427,0.534195,1.439177,...,1.0,4.0,24.0,13.0,18.0,7.0,2.0,3.0,2.0,4.0
4,EH38E0048077,EM10E1439220,brain,0.897288,1.258093,1.065049,1.156469,0.994539,0.024305,0.561160,...,10.0,69.0,245.0,87.0,102.0,23.0,7.0,80.0,27.0,27.0


In [12]:
from pathlib import Path
from Bio import SeqIO

root = Path(r"C:\Projects\eclare")

bed_dir = root / "data/encode"
genome_dir = root / "data/reference_genomes"

In [13]:
def read_bed(path, prefix):
    bed = pd.read_csv(path, sep="\t", header=None, comment="#")

    id_col = next(
        col for col in bed.columns[3:]
        if bed[col].astype(str).str.startswith(prefix).any()
    )

    result = bed[[0, 1, 2, id_col]].copy()
    result.columns = ["chrom", "start", "end", "ccre_id"]
    return result

In [14]:
human = read_bed(bed_dir / "GRCh38-cCREs.ELS.bed", "EH38E")
mouse = read_bed(bed_dir / "mm10-cCREs.ELS.bed", "EM10E")

In [16]:
human.head()

,chrom,start,end,ccre_id
0,chr1,79304,79545,EH38E2776519
1,chr1,190809,191058,EH38E2776525
2,chr1,191294,191644,EH38E2776526
3,chr1,191759,191935,EH38E2776527
4,chr1,271226,271468,EH38E2776529


Отфильтруем только нужные cCRE:

In [18]:
human = human[human.ccre_id.isin(df.human_id)].copy()
mouse = mouse[mouse.ccre_id.isin(df.mouse_id)].copy()

Извлечем последовательности из генома

In [19]:
def extract_sequences(table, genome_path):
    sequences = []

    with open(genome_path, "rb") as handle:
        genome = SeqIO.parse(handle, "twobit")

        for row in table.itertuples(index=False):
            chromosome = genome[row.chrom].seq
            assert 0 <= row.start < row.end <= len(chromosome)

            sequence = str(chromosome[row.start:row.end]).upper()
            sequences.append(sequence)

    return sequences

In [20]:
human["sequence"] = extract_sequences(human, genome_dir / "hg38.2bit")
mouse["sequence"] = extract_sequences(mouse, genome_dir / "mm10.2bit")

In [21]:
human = human.rename(columns={
    col: "human_id" if col == "ccre_id" else f"human_{col}"
    for col in human.columns
})

mouse = mouse.rename(columns={
    col: "mouse_id" if col == "ccre_id" else f"mouse_{col}"
    for col in mouse.columns
})

In [22]:
wide = wide.merge(human, on="human_id", validate="many_to_one")
wide = wide.merge(mouse, on="mouse_id", validate="many_to_one")

wide.head()

,human_id,mouse_id,tissue,human_zscore_ATAC,human_zscore_CTCF,human_zscore_DNase,human_zscore_H3K27ac,human_zscore_H3K4me3,mouse_zscore_ATAC,mouse_zscore_CTCF,...,mouse_n_measurements_H3K27ac,mouse_n_measurements_H3K4me3,human_chrom,human_start,human_end,human_sequence,mouse_chrom,mouse_start,mouse_end,mouse_sequence
0,EH38E0048077,EM10E1439220,adipose,NaN,NaN,1.008100,-0.018593,0.474738,NaN,NaN,...,1.0,1.0,chr6,35458191,35458520,GAGTATCTTTGCCCTGCTCTGTCTGCTGTCCAGTTGCTTGAGACAG...,chr17,28317175,28317409,AAGCCCCTAGTGGACCTGCAGTTTCTTCATGAATGTAGTCCCAGCG...
1,EH38E0048077,EM10E1439220,adrenal gland,NaN,NaN,0.950425,NaN,NaN,NaN,NaN,...,NaN,NaN,chr6,35458191,35458520,GAGTATCTTTGCCCTGCTCTGTCTGCTGTCCAGTTGCTTGAGACAG...,chr17,28317175,28317409,AAGCCCCTAGTGGACCTGCAGTTTCTTCATGAATGTAGTCCCAGCG...
2,EH38E0048077,EM10E1439220,blood,1.644916,2.003889,1.353530,1.340279,1.355181,0.412561,0.652838,...,1.0,1.0,chr6,35458191,35458520,GAGTATCTTTGCCCTGCTCTGTCTGCTGTCCAGTTGCTTGAGACAG...,chr17,28317175,28317409,AAGCCCCTAGTGGACCTGCAGTTTCTTCATGAATGTAGTCCCAGCG...
3,EH38E0048077,EM10E1439220,bone marrow,0.725278,0.999385,0.848175,1.302290,0.547427,0.534195,1.439177,...,2.0,4.0,chr6,35458191,35458520,GAGTATCTTTGCCCTGCTCTGTCTGCTGTCCAGTTGCTTGAGACAG...,chr17,28317175,28317409,AAGCCCCTAGTGGACCTGCAGTTTCTTCATGAATGTAGTCCCAGCG...
4,EH38E0048077,EM10E1439220,brain,0.897288,1.258093,1.065049,1.156469,0.994539,0.024305,0.561160,...,27.0,27.0,chr6,35458191,35458520,GAGTATCTTTGCCCTGCTCTGTCTGCTGTCCAGTTGCTTGAGACAG...,chr17,28317175,28317409,AAGCCCCTAGTGGACCTGCAGTTTCTTCATGAATGTAGTCCCAGCG...


In [25]:
signal_columns = [
    "human_zscore_DNase",
    "human_zscore_H3K27ac",
    "mouse_zscore_DNase",
    "mouse_zscore_H3K27ac",
]

measured = wide[signal_columns].notna().all(axis=1)
high_signal = wide[signal_columns].gt(1.64).all(axis=1)

active_pairs = wide.loc[measured & high_signal].copy()
active_pairs.head()

,human_id,mouse_id,tissue,human_zscore_ATAC,human_zscore_CTCF,human_zscore_DNase,human_zscore_H3K27ac,human_zscore_H3K4me3,mouse_zscore_ATAC,mouse_zscore_CTCF,...,mouse_n_measurements_H3K27ac,mouse_n_measurements_H3K4me3,human_chrom,human_start,human_end,human_sequence,mouse_chrom,mouse_start,mouse_end,mouse_sequence
36,EH38E0095409,EM10E1564830,heart,3.434156,1.328413,4.098152,2.817273,2.254305,1.677185,0.869383,...,10.0,10.0,chr1,66563483,66563828,GATCTCCTGACTGTGACCCTGGCTCTCCATCCACTCGGCTCCTATC...,chr4,102792148,102792475,AGAACACCATCAGGTCTGGGAGAACGAGCCCTAACAGGTCCCTTCT...
85,EH38E0133996,EM10E0451602,embryo,NaN,-0.036220,2.542946,2.872853,1.207283,NaN,0.243336,...,8.0,11.0,chr1,202954396,202954726,CACTTCTATGGGGTGGATCACTCACAAACAATGGAGAATTACAAGT...,chr1,134419272,134419487,CCCATTTGAGAGGAAAGGTGTTTCCTCTAAAATGCATAAGGGTTTA...
108,EH38E0134064,EM10E0451518,brain,1.186497,1.483916,3.329173,2.582384,1.164882,1.857880,0.381074,...,27.0,27.0,chr1,203102415,203102687,TGAAATCACAGATGCACAAATTATCAGAACTGGGAAGAATCTTAGG...,chr1,134274421,134274693,GAGAAACACAGAGAGAGTGAGAGAGGTGTCCATTTGCGACCATCAT...
218,EH38E0193584,EM10E1080411,heart,0.785844,1.497241,1.653091,2.858774,0.795013,1.322817,0.485297,...,10.0,10.0,chr10,110795386,110795567,GCTCCCCTTTGGGAAAGTGCTTGACCTCTGGACATTGCAGAGTGGG...,chr19,53831043,53831381,CCAGGTGGAACCATAAAAGGAAATGATAATTGACTCTAATTGTCCT...
244,EH38E0195486,EM10E1081202,heart,1.106814,0.859451,1.767988,1.722348,0.978002,2.051505,0.717501,...,10.0,10.0,chr10,114740080,114740281,AGAGAAAGAATGTTCGCCAGAAGTATCTAAGAGTGTGTCTAGGCTA...,chr19,57283114,57283305,AGAGGGCACAGCAAAATGGTAACAATCCTTCCAGGGTGCCGTGGAA...


In [29]:
active_pairs["tissue"].value_counts()

tissue
brain                7216
heart                3670
embryo               3394
kidney               3126
lung                 3026
liver                2117
connective tissue    1893
spleen               1613
blood                1579
bone marrow          1483
stomach              1198
thymus                876
epithelium            765
testis                692
adipose               217
Name: count, dtype: int64

In [30]:
active_pairs.to_parquet(
    root / "data/processed/baseline_with_dna.parquet",
    index=False,
)